# Applied Time Series — Assignment 2026
## Forecasting the U.S. charge-off rate on business loans (CORBLACBS)

*Thomas de Portzamparc & [Partner]* — M203 Financial Markets, Université Paris Dauphine-PSL

| Period | Dates | Use |
|---|---|---|
| In-sample | 2012Q1–2023Q4 (48 obs.) | estimation, model selection, tuning |
| Out-of-sample | 2024Q1–2025Q4 (8 obs.) | evaluation only |
| Update | sample up to 2026Q2, WEO April 2026 | forecast of 2026Q3 |

**How the notebook is organised.** Section 1 builds the database and the WEO scenario. Section 2 studies the in-sample period and, *before* looking at 2024–2025, fixes one common evaluation protocol for all models. Sections 3–4 forecast and evaluate 2024–2025. Section 5 updates the forecast for 2026Q3, section 6 opens the black boxes and section 7 is the AI usage statement.

**Golden rule.** Every selection step only reads data up to `IS_END` (2023Q4). The 2024–2025 values are used once, in section 4, to evaluate.

## 0. Setup
Relative paths only: the notebook expects a `data/` folder next to it. It downloads FRED directly and falls back on the CSV snapshot in `data/` if there is no internet connection.

In [ ]:
import itertools
import random
import warnings
from importlib.metadata import version
from pathlib import Path

import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # LightGBM and torch both ship an OpenMP runtime on macOS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.interpolate import CubicSpline

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ardl import ARDL, UECM, ardl_select_order
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from arch.unitroot import PhillipsPerron, ZivotAndrews

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

# one seed for everything (numpy, random, torch is seeded later through Darts)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = Path("data")

# the three periods of the assignment
IS_START, IS_END = pd.Period("2012Q1"), pd.Period("2023Q4")
OOS_END = pd.Period("2025Q4")
UPD_END, UPD_TARGET = pd.Period("2026Q2"), pd.Period("2026Q3")
H = 8  # out-of-sample horizon (2024Q1 -> 2025Q4)

for p in ["numpy", "pandas", "scipy", "statsmodels", "arch", "scikit-learn", "lightgbm", "torch", "darts", "shap", "timesfm"]:
    try:
        print(f"{p:<12} {version(p)}")
    except Exception:
        print(f"{p:<12} not installed")

## 1. Data
### 1.1 Building the database

| Variable | FRED code | Raw frequency | Quarterly conversion | Used in the models as | Why |
|---|---|---|---|---|---|
| $y_t$ | CORBLACBS | quarterly, % | — | level | the target; already a rate, no log needed |
| $g_t$ | GDPC1 | quarterly, bn chained 2017 \$ | — | $100\,\Delta_4 \ln GDP_t$ (y/y growth) | stationary, no base-year problem, directly comparable with WEO growth rates |
| $\pi_t$ | CPIAUCSL | monthly, index | **average of the 3 months** | $100\,\Delta_4 \ln CPI_t$ (y/y inflation) | the charge-off rate is a flow over the quarter, so we match it with the quarterly average; WEO inflation is also an annual-average concept |
| $u_t$ | UNRATE | monthly, % | **average of the 3 months** | level | WEO gives an annual mean; main cyclical driver of loan losses |
| $ca_t$ | IEABC and GDP | quarterly, \$ | — | current account in % of **nominal** GDP | IEABC is a nominal flow, so it must be divided by nominal GDP (not real GDP); WEO reports the same ratio |

Two practical points: FRED now publishes IEABC in **millions** of dollars (the course template was in billions), and nominal GDP is an annual rate, so the ratio is $100 \times \frac{IEABC/1000}{GDP/4}$. Average rather than end-of-period for the monthly series: the end-of-quarter value would throw away two months of information and is noisier.

In [ ]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}&cosd=2010-01-01"


def load_fred():
    """Download the six FRED series; if it fails, read the snapshot saved in data/."""
    try:
        get = lambda code: pd.read_csv(FRED_URL.format(code), index_col=0, parse_dates=True, na_values=".").iloc[:, 0]
        quarterly = pd.concat({c: get(c) for c in ["CORBLACBS", "GDPC1", "GDP", "IEABC"]}, axis=1)
        monthly = pd.concat({c: get(c) for c in ["CPIAUCSL", "UNRATE"]}, axis=1)
        quarterly.to_csv(DATA / "fred_quarterly.csv")
        monthly.to_csv(DATA / "fred_monthly.csv")
        print("FRED downloaded (snapshot in data/ updated).")
    except Exception as err:
        print(f"Download failed ({type(err).__name__}), using the snapshot in data/.")
        quarterly = pd.read_csv(DATA / "fred_quarterly.csv", index_col=0, parse_dates=True)
        monthly = pd.read_csv(DATA / "fred_monthly.csv", index_col=0, parse_dates=True)
    return quarterly, monthly


fred_q, fred_m = load_fred()

# monthly -> quarterly: average of the months of the quarter
monthly_to_q = fred_m.resample("QS").mean()
raw = fred_q.join(monthly_to_q, how="outer")
raw.index = raw.index.to_period("Q")

# a quarter with fewer than 3 months published is flagged (Oct-2025 was never released: US shutdown)
n_months = fred_m.resample("QS").count()
n_months.index = n_months.index.to_period("Q")
print("Quarters with missing months:\n", n_months[(n_months < 3).any(axis=1)].loc[:"2026Q2"])
print(raw.tail(6).round(2))

In [ ]:
def build_panel(raw):
    """Transform the raw FRED data into the five variables used in the models."""
    ieabc_bn = raw["IEABC"] / 1000 if raw["IEABC"].abs().median() > 5000 else raw["IEABC"]  # millions -> billions
    panel = pd.DataFrame(index=raw.index)
    panel["y"] = raw["CORBLACBS"]
    panel["g"] = 100 * np.log(raw["GDPC1"]).diff(4)
    panel["pi"] = 100 * np.log(raw["CPIAUCSL"]).diff(4)
    panel["u"] = raw["UNRATE"]
    panel["ca"] = 100 * ieabc_bn / (raw["GDP"] / 4)
    return panel


panel = build_panel(raw)
X = ["g", "pi", "u", "ca"]
NAMES = {"y": "Charge-off rate (%)", "g": "Real GDP growth y/y (%)", "pi": "CPI inflation y/y (%)",
         "u": "Unemployment rate (%)", "ca": "Current account (% GDP)"}

insample = panel.loc[IS_START:IS_END]
assert len(insample) == 48 and insample.notna().all().all()
insample.describe().T[["mean", "std", "min", "max"]].round(2)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
for ax, col in zip(axes.flat, ["y"] + X):
    s = panel[col].loc["2011Q1":"2026Q2"]
    ax.plot(s.index.to_timestamp(), s.values, color="k")
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31"), color="red", alpha=0.1)      # Covid
    ax.axvline(pd.Timestamp("2023-03-10"), color="purple", ls=":")                                    # SVB failure
    ax.axvspan(pd.Timestamp("2024-01-01"), pd.Timestamp("2025-12-31"), color="orange", alpha=0.12)  # out-of-sample
    ax.set_title(NAMES[col], fontsize=10)
axes.flat[-1].axis("off")
fig.suptitle("Red: Covid-19 (2020) — dotted: SVB failure (March 2023) — orange: out-of-sample 2024-2025", fontsize=10)
plt.tight_layout()
plt.show()

**What the plots show.**
* **Charge-off rate.** No trend over 2012–2023 (mean 0.32 %), but a clear cycle. It is stable around 0.3 % until 2019, peaks at 0.59 % in 2020Q2 (Covid), then collapses to 0.12 % in 2021Q4–2022Q1. Massive fiscal support (PPP loans, transfers) and rising asset prices kept firms alive. It then climbs back steadily: 0.42 % in 2023Q4 and 0.55–0.58 % in 2024–2025. The series is very persistent: quarter-to-quarter moves are small compared with its swings.
* **Covid is the dominant outlier in every covariate.** Unemployment jumps to 13 %, y/y GDP growth falls to −7.7 % and then rebounds to +11.8 % in 2021Q2. That rebound is a base effect of the y/y transformation: the 2020 drop comes back as a mirror spike four quarters later. With only 48 points, these 4–6 quarters carry a large share of the covariance between $y$ and the covariates, so they will drive the estimated effects.
* **Inflation** has its own regime in 2021–2022 (up to 8.2 %). Over this window it looks more like a level shift than noise around a mean.
* **Current account.** It drifts from about −2 % to −3.5/−4 % of GDP after 2020: another level shift.
* **SVB failure (March 2023).** There is no jump in the charge-off rate around it. The turmoil was a deposit/funding shock for regional banks, while business-loan losses kept rising at the same pace as before. So we do not add a dummy for it.

**Data quality.** October 2025 CPI and unemployment were never published (US government shutdown), so 2025Q4 is the average of 2 months instead of 3. It only matters for the out-of-sample evaluation and the 2026 update.

### 1.2 Quarterly scenario from the annual WEO projections (needed for Question 3.1)

The WEO gives **annual** numbers, and an annual number can mean three different things:

| WEO concept | Constraint on the four quarters | Where the annual value is attached |
|---|---|---|
| annual average (GDP level, CPI level, unemployment) | mean of the 4 quarters = annual value | middle of the year |
| annual sum (a flow in \$, e.g. the current account in \$ bn) | sum of the 4 quarters = annual value | middle of the year, value × 4 |
| end of period (a stock) | Q4 value = annual value | Q4 |

All the variables we use are annual averages (the current-account ratio is BCA/NGDP for the year, which is very close to the average of the quarterly ratios).

**Method.** We fit a natural cubic spline through (i) the last four *observed* quarters and (ii) one knot per projected year placed in the middle of the year. The spline passes through the observed points, so the path is connected smoothly to 2023Q4. A knot placed at the annual value does not give a quarterly average exactly equal to it (the curve bends), so we correct the knots a few times until the four quarters average to the WEO figure. We check the result in the table below.

**Reconciling WEO and FRED.** The WEO level of real GDP is not in the same base year/vintage as FRED's chained 2017 dollars. So we never use WEO *levels*: we take WEO *growth rates* for GDP and CPI, chain them on the FRED annual average of the last complete year, spline the implied quarterly levels, and compute our y/y growth from these levels. Unemployment and the current-account ratio have the same units in both sources and are splined directly.

In [ ]:
def spline_quarterly(hist, targets):
    """Quarterly path after hist whose 4 quarters average to targets[year] for each projected year.

    hist    : observed quarterly values up to the forecast origin (PeriodIndex)
    targets : {year: annual WEO value (annual average)}
    """
    years = sorted(targets)
    future = pd.period_range(hist.index[-1] + 1, pd.Period(f"{years[-1]}Q4"), freq="Q")
    t_hist = np.array([p.ordinal for p in hist.index[-4:]], dtype=float)
    t_future = np.array([p.ordinal for p in future], dtype=float)
    # one knot per year, in the middle of the quarters still unknown that year
    t_knots = np.array([t_future[[p.year == y for p in future]].mean() for y in years])
    knot_values = np.array([targets[y] for y in years], dtype=float)  # first guess

    for _ in range(100):
        spline = CubicSpline(np.r_[t_hist, t_knots], np.r_[hist.values[-4:], knot_values], bc_type="natural")
        path = pd.Series(spline(t_future), index=future)
        full = pd.concat([hist, path])
        errors = np.array([full[[p.year == y for p in full.index]].mean() - targets[y] for y in years])
        n_unknown = np.array([sum(p.year == y for p in future) for y in years])
        knot_values -= errors * 4 / n_unknown  # move each knot by the missing amount
        if np.abs(errors).max() < 1e-10:
            break
    return path


def weo_scenario(raw, weo, origin, end):
    """Quarterly path of g, pi, u, ca implied by one WEO vintage, from origin+1 to end."""
    hist_raw = raw.loc[:origin]
    years = [y for y in weo.index if y >= (origin + 1).year]
    last_full_year = origin.year if origin.quarter == 4 else origin.year - 1
    levels = {}
    for col, code in [("GDPC1", "NGDP_RPCH"), ("CPIAUCSL", "PCPIPCH")]:
        level = hist_raw[col].dropna()
        annual = level.groupby(level.index.year).mean().loc[last_full_year]
        targets = {}
        for y in range(last_full_year + 1, max(years) + 1):
            annual *= 1 + weo.loc[y, code] / 100  # chain WEO growth on the FRED annual average
            targets[y] = annual
        levels[col] = pd.concat([level, spline_quarterly(level, {y: targets[y] for y in years})])
    levels = pd.DataFrame(levels)

    scen = pd.DataFrame(index=pd.period_range(origin + 1, end, freq="Q"))
    scen["g"] = 100 * np.log(levels["GDPC1"]).diff(4)
    scen["pi"] = 100 * np.log(levels["CPIAUCSL"]).diff(4)
    hist_panel = build_panel(hist_raw)
    for col, code in [("u", "LUR"), ("ca", "BCA_NGDPD")]:
        scen[col] = spline_quarterly(hist_panel[col].dropna(), {y: weo.loc[y, code] for y in years})
    return scen.loc[origin + 1:end], levels


weo = pd.read_csv(DATA / "weo_vintages.csv")
weo23 = weo[weo.vintage == "2023-10"].set_index("year")
print(weo23[["NGDP_RPCH", "PCPIPCH", "LUR", "BCA_NGDPD"]])

scen23, levels23 = weo_scenario(raw, weo23, IS_END, OOS_END)

# check: do the four quarters give back the WEO annual figures?
check = pd.DataFrame(index=[2024, 2025])
for y in check.index:
    gdp, cpi = levels23["GDPC1"], levels23["CPIAUCSL"]
    check.loc[y, "GDP growth WEO"] = weo23.loc[y, "NGDP_RPCH"]
    check.loc[y, "GDP growth spline"] = 100 * (gdp.loc[str(y)].mean() / gdp.loc[str(y - 1)].mean() - 1)
    check.loc[y, "inflation WEO"] = weo23.loc[y, "PCPIPCH"]
    check.loc[y, "inflation spline"] = 100 * (cpi.loc[str(y)].mean() / cpi.loc[str(y - 1)].mean() - 1)
    check.loc[y, "unemployment WEO"] = weo23.loc[y, "LUR"]
    check.loc[y, "unemployment spline"] = scen23.loc[str(y), "u"].mean()
    check.loc[y, "CA/GDP WEO"] = weo23.loc[y, "BCA_NGDPD"]
    check.loc[y, "CA/GDP spline"] = scen23.loc[str(y), "ca"].mean()
check.T.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, X):
    real = panel[col].loc["2021Q1":"2025Q4"]
    ax.plot(real.index.to_timestamp(), real.values, color="k", label="realised (FRED)")
    ax.plot(scen23.index.to_timestamp(), scen23[col].values, "o-", ms=3, label="WEO Oct-2023, splined")
    ax.axvline(IS_END.end_time, color="grey", ls="--")
    ax.set_title(NAMES[col], fontsize=9)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

scenario_error = scen23 - panel.loc[scen23.index, X]
scenario_error.round(2).add_prefix("WEO - realised: ")

**Consistency.** The check table confirms that the four quarters of every projected year average *exactly* to the WEO annual figure. The spline also starts from the last observed quarter, so there is no jump at 2023Q4.

**Was the October 2023 WEO a good scenario?**
* **Growth was the big miss.** The WEO expected a slowdown to 1.5 % in 2024, but the US grew about 2.9 %. The splined y/y growth even dips to ≈0.4 % in 2024Q4. This is a **carry-over effect**: 2023 ended well above its annual average, so the level at the end of 2023 already implies about 1.2 % growth for 2024. A 1.5 % annual average then leaves almost no growth *within* 2024. The dip is the honest consequence of chaining a 2023 forecast on today's revised 2023 data. If we had used the WEO's own (lower) 2023 level, the path would be smoother. But then the scenario would not connect to the last observed quarter, which the question asks for.
* **Unemployment was too optimistic** by 0.2–0.6 pp at the end of 2025 (WEO 3.9 %, realised 4.3–4.5 %).
* **Inflation** is close: the WEO is 0.2–0.5 pp too low.
* **Current account** is off by 1.3 pp of GDP. The realised series includes the 2025Q1 import surge ahead of the tariffs (−5.8 % of GDP), which no forecaster could see in 2023.

So the WEO was too pessimistic on growth and too optimistic on unemployment. For charge-offs, these two errors push in opposite directions. This will matter in Question 4.2.

## 2. In-sample modelling (2012Q1–2023Q4)
### 2.1 Preliminary analysis: stationarity
We use the three tests of the course with complementary null hypotheses — ADF (H0: unit root, autoregressive correction), Phillips–Perron (H0: unit root, non-parametric correction), KPSS (H0: stationarity) — plus Zivot–Andrews (H0: unit root, against stationarity with one break), because the sample contains Covid. Deterministic part: a constant, since none of the series has a credible trend over 12 years.

In [ ]:
def unit_root_tests(series):
    s = series.dropna()
    adf = adfuller(s, regression="c", autolag="AIC")
    pp = PhillipsPerron(s, trend="c")
    kp = kpss(s, regression="c", nlags="auto")
    za = ZivotAndrews(s, trend="c", max_lags=4)  # default lag search is infeasible with 48 points
    return {"ADF stat": adf[0], "ADF p": adf[1], "PP p": pp.pvalue, "KPSS stat": kp[0], "KPSS p": kp[1], "ZA p": za.pvalue}


rows = {}
for col in ["y"] + X:
    rows[col] = unit_root_tests(insample[col])
    rows["Δ" + col] = {k: v for k, v in unit_root_tests(insample[col].diff()).items()}
pd.DataFrame(rows).T.round(3)

**How much can a unit-root test say with 48 quarters?** We simulate stationary AR(1) processes with 48 observations and count how often the ADF test (5 %) rejects the unit root.

In [ ]:
def adf_rejection_rate(rho, T=48, n_sim=1000):
    rng = np.random.default_rng(SEED)
    rejections = 0
    for _ in range(n_sim):
        e = rng.standard_normal(T + 50)
        x = np.zeros(T + 50)
        for t in range(1, T + 50):
            x[t] = rho * x[t - 1] + e[t]
        rejections += adfuller(x[50:], regression="c", maxlag=1, autolag=None)[1] < 0.05
    return rejections / n_sim


pd.Series({rho: adf_rejection_rate(rho) for rho in [1.0, 0.95, 0.9, 0.8, 0.7]}, name="P(reject unit root), T=48").to_frame().T

**Reading the tests.**

| Series | Evidence | Treatment |
|---|---|---|
| $y$ | ADF rejects (p = 0.002), KPSS does not reject stationarity, PP and ZA are borderline | I(0) but very persistent |
| $g$ | all tests agree | stationary (not surprising for a growth rate) |
| $\pi$ | ADF and PP cannot reject a unit root, KPSS rejects stationarity at 5 % | locally non-stationary: the 2021–22 surge looks like a level shift |
| $u$ | ADF and PP reject at 5 %, ZA rejects strongly (p = 0.003) | stationary around the Covid break |
| $ca$ | ADF/PP do not reject, KPSS rejects, but ZA rejects (p < 0.001) | stationary with one break in 2020 |

On $y$, PP disagrees with ADF because it only corrects an AR(1) regression non-parametrically, while the series has a strong AR(2)-type cycle that ADF's extra lags capture. The $ca$ case is the classic Perron (1989) result: an unmodelled break makes ADF look like a unit root.

**Power with 48 observations.** The simulation is sobering. If the truth is a stationary AR(1) with $\rho = 0.9$, ADF detects it only **13 %** of the time; with $\rho = 0.95$, only 9 %, barely above the 5.8 % size. So "we cannot reject a unit root" is very weak evidence here. We let economics break the tie.

**A rate is bounded.** The charge-off rate lives in $[0, 100]$ and cannot wander like a random walk forever, so it is I(0) by construction, even if a short sample looks persistent. The same holds for the unemployment rate. The bound has a second consequence: near zero (0.12 % in 2021) the rate cannot fall much further, so the dynamics are asymmetric. Our linear Gaussian models ignore this, and their intervals could in principle go below zero. A logit transformation would fix it, but the series is far from the bound after 2022, so we keep levels for interpretability.

**Decision.** All models are in levels (no differencing of $y$). $\pi$ is kept in levels despite the tests, because it is defined like the WEO variable. The ARDL bounds test (2.4) is valid whether the regressors are I(0) or I(1), which protects us against a wrong call.

### A common evaluation protocol (used from 2.2 to 2.7)
To compare models as different as an ARIMA and a neural network, we need one rule that is the same for everybody. We use an **expanding-window pseudo out-of-sample backtest inside the in-sample period**:
* forecast origins 2018Q4, 2019Q1, …, 2022Q4 (17 origins);
* at each origin the model is re-estimated on 2012Q1–origin and forecasts up to 8 quarters ahead (the horizon of Question 3), keeping only targets ≤ 2023Q4;
* over the horizon the covariates take their **realised** values, so the backtest measures model error only (in Question 3 they will come from the WEO).

**Metric: MASE** = MAE divided by the in-sample MAE of the one-step naive forecast. It is scale-free (comparable across horizons and with the 2026 update, unlike RMSE), it does not explode when the rate is close to 0 (unlike MAPE: $y$ goes down to 0.12 %), and it is easy to read: MASE = 1 means an error as large as a typical one-quarter change. RMSE and Theil's U (RMSE relative to the random walk) are reported as well; intervals are judged by their empirical coverage.

Every model is written as a function `f(train, x_future, h)` that returns a table with the forecast (`mean`) and an 80 % interval (`lo`, `hi`).

In [ ]:
BT_ORIGINS = pd.period_range("2018Q4", "2022Q4", freq="Q")
ALPHA = 0.20                          # 80 % prediction intervals
Z80 = stats.norm.ppf(1 - ALPHA / 2)
MASE_SCALE = insample["y"].diff().abs().mean()


def backtest(model, data=panel, origins=BT_ORIGINS, h=H, end=IS_END):
    """Expanding-window forecasts: re-estimate at each origin, forecast h steps, store the errors."""
    rows = []
    for origin in origins:
        steps = min(h, (end - origin).n)
        train = data.loc[IS_START:origin]
        x_future = data.loc[origin + 1:origin + steps, X]   # realised covariates over the horizon
        fc = model(train, x_future, steps)
        for k, (target, row) in enumerate(fc.iterrows(), start=1):
            rows.append({"origin": origin, "target": target, "h": k, "mean": row["mean"],
                         "lo": row["lo"], "hi": row["hi"], "y": data.loc[target, "y"]})
    out = pd.DataFrame(rows)
    out["error"] = out["y"] - out["mean"]
    return out


def scores(bt):
    e = bt["error"]
    coverage = ((bt["y"] >= bt["lo"]) & (bt["y"] <= bt["hi"])).mean() if bt["lo"].notna().all() else np.nan
    return pd.Series({"MASE": e.abs().mean() / MASE_SCALE, "RMSE": np.sqrt((e ** 2).mean()),
                      "MAE": e.abs().mean(), "Bias": e.mean(), "Coverage 80%": coverage})


def mase_by_h(bt):
    return bt.groupby("h")["error"].apply(lambda e: e.abs().mean() / MASE_SCALE)


def output(mean, lo, hi, index):
    return pd.DataFrame({"mean": np.asarray(mean, float), "lo": np.asarray(lo, float), "hi": np.asarray(hi, float)}, index=index)


def naive(train, x_future, h):
    """Random walk: the last value, with a sqrt(h) interval."""
    sigma = train["y"].diff().std()
    k = np.arange(1, h + 1)
    last = train["y"].iloc[-1]
    return output(np.repeat(last, h), last - Z80 * sigma * np.sqrt(k), last + Z80 * sigma * np.sqrt(k), x_future.index)


backtests = {"Naive (RW)": backtest(naive)}

### 2.2 ARIMA (univariate benchmark)
**(a) Model.** $\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t$, with $\varepsilon_t$ white noise (constant only if $d=0$).
**(b) Estimation.** Exact Gaussian maximum likelihood with the Kalman filter (`SARIMAX`), stationarity and invertibility imposed.
**(c) Selection.** Box–Jenkins: ACF/PACF to get an idea of the orders, then a grid $p, q \le 3$, $d \in \{0, 1\}$. BIC is only comparable for a given $d$ (the dependent variable changes with $d$), so it is used to build a short list (2 best models for each $d$). **"Most accurate" = lowest backtest MASE over horizons 1 to 8**, because the goal is to forecast 8 quarters ahead, not to fit the past. The model must also pass the residual checks.
**(d) Covariates.** None. **(e) Forecasts.** Iterated conditional expectations; Gaussian interval with variance $\sigma^2\sum_{j<h}\psi_j^2$ (parameter uncertainty ignored).
**(f) Limits.** Pure extrapolation of the past dynamics: it cannot anticipate a turn of the credit cycle.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 5.5))
plot_acf(insample["y"], lags=16, ax=axes[0, 0], title="ACF of y")
plot_pacf(insample["y"], lags=16, ax=axes[0, 1], method="ywm", title="PACF of y")
plot_acf(insample["y"].diff().dropna(), lags=16, ax=axes[1, 0], title="ACF of Δy")
plot_pacf(insample["y"].diff().dropna(), lags=16, ax=axes[1, 1], method="ywm", title="PACF of Δy")
plt.tight_layout()
plt.show()

In [ ]:
def fit_arima(y, order, exog=None):
    trend = "c" if order[1] == 0 else "n"
    return SARIMAX(y, exog=exog, order=order, trend=trend).fit(disp=False, maxiter=500)


rows = []
for d, p, q in itertools.product([0, 1], range(4), range(4)):
    res = fit_arima(insample["y"], (p, d, q))
    rows.append({"order": (p, d, q), "d": d, "BIC": res.bic, "AIC": res.aic,
                 "Ljung-Box(8) p": acorr_ljungbox(res.resid.iloc[d + p:], lags=[8], model_df=p + q)["lb_pvalue"].iloc[0]})
grid = pd.DataFrame(rows)
shortlist = pd.concat([grid[grid.d == d].nsmallest(2, "BIC") for d in (0, 1)])
shortlist

In [ ]:
def arima_model(order, cols=None):
    """Build a forecasting function for an ARIMA(order) model, with optional covariates (SARIMAX)."""
    def f(train, x_future, h):
        res = fit_arima(train["y"], order, exog=train[cols] if cols else None)
        fc = res.get_forecast(h, exog=x_future[cols] if cols else None)
        ci = fc.conf_int(alpha=ALPHA)
        return output(fc.predicted_mean, ci.iloc[:, 0], ci.iloc[:, 1], x_future.index)
    return f


for order in shortlist["order"]:
    backtests[f"ARIMA{order}"] = backtest(arima_model(order))
shortlist["backtest MASE"] = [scores(backtests[f"ARIMA{o}"])["MASE"] for o in shortlist["order"]]
shortlist["MASE h=1"] = [mase_by_h(backtests[f"ARIMA{o}"]).iloc[0] for o in shortlist["order"]]
shortlist["MASE h=8"] = [mase_by_h(backtests[f"ARIMA{o}"]).iloc[-1] for o in shortlist["order"]]
print("Naive backtest MASE:", round(scores(backtests["Naive (RW)"])["MASE"], 3))
shortlist.round(3)

In [ ]:
ARIMA_ORDER = shortlist.sort_values("backtest MASE")["order"].iloc[0]
arima_res = fit_arima(insample["y"], ARIMA_ORDER)
print("Retained:", ARIMA_ORDER)
print(arima_res.summary().tables[1])
inv_roots = 1 / arima_res.arroots
print("Inverse AR roots (modulus):", np.round(np.abs(inv_roots), 3))
# a complex pair of roots = a damped stochastic cycle; its period is 2*pi / angle
angle = np.abs(np.angle(inv_roots)).max()
print(f"Implied cycle period: {2 * np.pi / angle:.1f} quarters")


def residual_checks(resid, n_params):
    e = resid.dropna()
    lb = acorr_ljungbox(e, lags=[6, 8, 12], model_df=n_params)["lb_pvalue"]  # lags > n_params, otherwise df = 0
    jb_stat, jb_p, skew, kurt = jarque_bera(e)
    return pd.Series({"LB(6) p": lb.iloc[0], "LB(8) p": lb.iloc[1], "LB(12) p": lb.iloc[2],
                      "JB p": jb_p, "skewness": skew, "kurtosis": kurt, "ARCH-LM(4) p": het_arch(e, nlags=4)[1]})


# the first residuals only reflect the initialisation of the Kalman filter: we drop p of them
p_ar = ARIMA_ORDER[0] + ARIMA_ORDER[1]
resid = arima_res.resid.iloc[p_ar:]
print(residual_checks(resid, ARIMA_ORDER[0] + ARIMA_ORDER[2]).round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(resid.index.to_timestamp(), resid.values, width=60, color="k")
axes[0].set_title("Residuals")
plot_acf(resid, lags=12, ax=axes[1], title="ACF of the residuals")
plt.tight_layout()
plt.show()

**Identification.** The ACF of $y$ decays slowly in waves, and the PACF has a large positive spike at lag 1 and a negative one at lag 2. This is the signature of an AR(2) with complex roots, i.e. a cycle. After differencing, the ACF of $\Delta y$ is much smaller but not white noise.

**Selection.** For $d = 0$, BIC prefers ARMA(2,2), then ARMA(3,1); for $d = 1$ it prefers (0,1,2). Our accuracy criterion decides: ARMA(3,1) has a backtest MASE of 2.16, against 2.59 for ARMA(2,2), 3.42 for ARIMA(0,1,2) and 3.49 for the random walk. The differenced models are much worse at long horizons (MASE 4.8 at $h = 8$): they forget that the rate reverts to its mean. This confirms the I(0) choice of 2.1.

**Estimates.** $\phi_1 = 1.69$, $\phi_2 = -0.67$ and $\theta_1 = -0.94$ are highly significant. $\phi_3$ is not ($p = 0.22$); we keep it because the order was chosen on forecast accuracy, not on t-stats.
* The AR part has a pair of complex inverse roots with modulus **0.987**, which means a slowly damped stochastic cycle with a period of about **18 quarters (4.5 years)**. This is the credit cycle.
* The implied mean is $0.0443 / (1 - \sum\phi_i) \approx 0.33\,\%$, close to the sample mean.

**Diagnostics.**
* No residual autocorrelation (Ljung–Box p-values well above 5 % at 6, 8 and 12 lags) and no ARCH effect.
* Normality is strongly rejected (skewness 1.3, kurtosis 6) because of the 2020 shocks. The ML estimates stay consistent (quasi-ML), but the Gaussian intervals are too thin in the tails: in the backtest the 80 % interval only covers **56 %** of the outcomes.

### 2.3 ARIMAX
`SARIMAX` estimates a **regression with ARIMA errors**:
$$y_t = c + \beta' x_t + u_t, \qquad \phi(L)(1-L)^d u_t = \theta(L)\varepsilon_t,$$
while the **ARMAX** is $\phi(L) y_t = c + \beta' x_t + \theta(L)\varepsilon_t$.

The difference: multiply the first model by $\phi(L)$ (case $d=0$) to get $\phi(L) y_t = \phi(1)c + \beta'\phi(L) x_t + \theta(L)\varepsilon_t$. So the regression with ARMA errors *is* an ARMAX, but one where the lags of $x$ are forced to follow the same polynomial $\phi(L)$ as $y$ (common factor restriction).

*Interpretation of β.* In the regression with ARIMA errors, $\beta$ is the effect of $x_t$ on the level of $y_t$: a permanent change of $x$ moves $y$ at once and fully (impact = long run); the dynamics are only in the error. In the ARMAX, $\beta$ is only the **impact** effect; it then spreads through the AR part and the long-run effect is $\beta / (1 - \sum \phi_i)$.

**(a)** Regression with ARMA errors, same ARMA as in 2.2. **(b)** Exact ML (Kalman filter). **(c)** We estimate all the $2^4 - 1 = 15$ subsets of $\{g, \pi, u, ca\}$, contemporaneous or lagged one quarter, rank them by BIC, and check the best ones in the backtest. **(d)** The covariates enter linearly in the mean; out of sample they must be given (WEO path). **(e)** $\hat y_{T+h} = \hat c + \hat\beta' x_{T+h} + \hat u_{T+h|T}$; the interval only reflects the error process, **not** the uncertainty on the scenario. **(f)** Static, linear effect; needs a forecast of the covariates.

In [ ]:
rows = []
for k in range(1, 5):
    for cols in itertools.combinations(X, k):
        cols = list(cols)
        for lag in (0, 1):
            data = panel.copy()
            data[cols] = panel[cols].shift(lag)
            res = fit_arima(data.loc[IS_START:IS_END, "y"], ARIMA_ORDER, exog=data.loc[IS_START:IS_END, cols])
            rows.append({"covariates": "+".join(cols), "lag": lag, "BIC": res.bic,
                         "max p-value of β": res.pvalues[cols].max(), "cols": cols})
arimax_grid = pd.DataFrame(rows).sort_values("BIC").head(5).reset_index(drop=True)

# backtest of the 5 best specifications (all of them turn out to be contemporaneous)
for i, row in arimax_grid.iterrows():
    data = panel.copy()
    data[row.cols] = panel[row.cols].shift(row.lag)
    arimax_grid.loc[i, "backtest MASE"] = scores(backtest(arima_model(ARIMA_ORDER, row.cols), data))["MASE"]
arimax_grid.drop(columns="cols").round(3)

In [ ]:
ARIMAX_COLS = arimax_grid["cols"].iloc[0]   # rule fixed ex ante: best BIC, the backtest is a check
arimax_res = fit_arima(insample["y"], ARIMA_ORDER, exog=insample[ARIMAX_COLS])
print(arimax_res.summary().tables[1])
print(residual_checks(arimax_res.resid.iloc[p_ar:], ARIMA_ORDER[0] + ARIMA_ORDER[2]).round(3).to_string())
backtests[f"ARIMAX[{'+'.join(ARIMAX_COLS)}]"] = backtest(arima_model(ARIMA_ORDER, ARIMAX_COLS))

# same covariates in an ARMAX estimated by OLS: y_t on its own lags and x_t
n_lags = ARIMA_ORDER[0]
lags = pd.concat({f"y.L{i}": panel["y"].shift(i) for i in range(1, n_lags + 1)}, axis=1)
armax_X = sm.add_constant(pd.concat([lags, panel[ARIMAX_COLS]], axis=1)).loc[IS_START:IS_END]
armax = sm.OLS(insample["y"], armax_X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(armax.summary().tables[1])
phi_sum = armax.params[lags.columns].sum()
for c in ARIMAX_COLS:
    print(f"{c}: β regression with ARMA errors = {arimax_res.params[c]:.4f} | "
          f"β ARMAX (impact) = {armax.params[c]:.4f} | ARMAX long run = {armax.params[c] / (1 - phi_sum):.4f}")

**Which variables.** Among the 30 specifications, BIC ranks first unemployment alone, then growth alone, then $g + u$, all **contemporaneous**: a one-quarter lag never enters the top 5. Inflation and the current account only appear in worse specifications, with p-values of 0.84–0.95. Following our rule (best BIC), we keep $u_t$. In the backtest, $g$ or $g + u$ would be marginally better (1.81 and 1.79 vs 1.86). That difference is far too small to justify breaking the rule after looking at the result.

**Interpretation of β.**
* In the regression with ARMA errors, $\hat\beta_u = 0.013$ ($p = 0.22$): one more point of unemployment goes with a charge-off rate 1.3 bp higher, at once and permanently.
* In the ARMAX estimated by OLS, the impact coefficient is $-0.001$ ($p = 0.76$) and the long-run effect is $-0.005$.

The two numbers differ because they answer different questions. In the ARMAX, $y_{t-1}, y_{t-2}, y_{t-3}$ already explain almost everything, so $\beta$ only measures the *surprise* effect of $u_t$ given the past of $y$. In the regression with ARMA errors, $\beta$ measures the co-movement of the *levels* of $y$ and $u$, with the persistence pushed into the error.

**Economic reading.** Both say that the unemployment–charge-off link is weak over 2012–2023. Covid explains why: unemployment reached 13 %, yet charge-offs fell to their lowest level a year later, because government support replaced the usual default channel. With 48 points, this one episode is enough to flatten the estimated relation.

**Diagnostics.** Residuals are white noise (Ljung–Box) and show no ARCH effect. Normality is still rejected at 5 % ($p = 0.017$), but much less strongly than for the ARIMA: $u$ absorbs part of the 2020 shock. In the backtest the ARIMAX improves on the ARIMA (MASE 1.86 vs 2.16), mainly at short horizons ($h = 1$: 1.00 vs 1.14).

### 2.4 ARDL
**(a) Model.** $y_t = c + \sum_{i=1}^{p}\phi_i y_{t-i} + \sum_{j}\sum_{l=0}^{q_j}\beta_{j,l}\, x_{j,t-l} + \varepsilon_t$.
**(b) Estimation.** OLS (all candidate models on the same sample).
**(c) Orders.** `ardl_select_order` with $p \le 4$, $q_j \le 4$, a covariate can also be dropped, criterion **BIC** (AIC is shown to see what over-fitting looks like with 48 points).
**(d) Covariates.** Each one has its own distributed lag. **(e) Forecasts.** Dynamic (iterated) prediction with the scenario for $x$; Gaussian interval from the $\psi$-weights of $1/\phi(L)$, conditional on $x$.
**(f) Limits.** White-noise errors: the persistence of $y$ must be absorbed by its own lags; a long-run relation is hard to identify on 12 years with one big shock.

**Multipliers.** Short run (impact): $\beta_{j,0}$. Long run: $\theta_j = \sum_l \beta_{j,l} / (1 - \sum_i \phi_i)$ (standard error by the delta method).
**Error-correction form** (UECM): $\Delta y_t = c + \pi_y y_{t-1} + \sum_j \pi_j x_{j,t-1} + \text{short-run terms} + \varepsilon_t$, with $\pi_y = -(1-\sum\phi_i)$ and $\theta_j = -\pi_j / \pi_y$.
**Bounds test** (Pesaran, Shin and Smith, 2001): F-test of $\pi_y = \pi_1 = \dots = 0$; valid whether the regressors are I(0) or I(1), compared with the lower (all I(0)) and upper (all I(1)) critical values.

In [ ]:
sel_bic = ardl_select_order(insample["y"], 4, insample[X], 4, ic="bic", trend="c")
sel_aic = ardl_select_order(insample["y"], 4, insample[X], 4, ic="aic", trend="c")
# careful: dl_lags is a dict {variable: lags}; covariates dropped by BIC are simply absent
print("BIC:", sel_bic.model.ar_lags, sel_bic.model.dl_lags)
print("AIC:", sel_aic.model.ar_lags, sel_aic.model.dl_lags)

ARDL_P = max(sel_bic.model.ar_lags)
ARDL_Q = {var: max(l) for var, l in sel_bic.model.dl_lags.items()}
ardl_res = ARDL(insample["y"], ARDL_P, insample[list(ARDL_Q)], ARDL_Q, trend="c").fit()
print(ardl_res.summary().tables[1])
print(residual_checks(ardl_res.resid, 0).round(3).to_string())

In [ ]:
def psi_weights(phi, h):
    """MA(inf) weights of 1/phi(L), used for the ARDL forecast variance."""
    psi = np.zeros(h)
    psi[0] = 1
    for j in range(1, h):
        psi[j] = sum(phi[i] * psi[j - 1 - i] for i in range(min(len(phi), j)))
    return psi


def ardl_model(p, q):
    cols = list(q)
    def f(train, x_future, h):
        res = ARDL(train["y"], p, train[cols], q, trend="c").fit()
        n = len(train)
        mean = res.predict(start=n, end=n + h - 1, dynamic=True, exog_oos=x_future[cols]).values
        phi = [res.params.get(f"y.L{i}", 0) for i in range(1, p + 1)]
        se = np.sqrt(res.sigma2 * np.cumsum(psi_weights(phi, h) ** 2))
        return output(mean, mean - Z80 * se, mean + Z80 * se, x_future.index)
    return f


ARDL_AIC = (max(sel_aic.model.ar_lags), {v: max(l) for v, l in sel_aic.model.dl_lags.items()})
backtests["ARDL (BIC)"] = backtest(ardl_model(ARDL_P, ARDL_Q))
ardl_aic_bt = backtest(ardl_model(*ARDL_AIC))
pd.DataFrame({"ARDL (BIC)": scores(backtests["ARDL (BIC)"]), "ARDL (AIC)": scores(ardl_aic_bt)}).T.round(3)

In [ ]:
def long_run_multipliers(res, p, q):
    """Long-run multiplier sum(beta)/(1-sum(phi)) with a delta-method standard error."""
    b, V = res.params, res.cov_params()
    phis = [f"y.L{i}" for i in range(1, p + 1)]
    rows = {}
    for var, qj in q.items():
        betas = [f"{var}.L{l}" for l in range(qj + 1)]
        den, num = 1 - b[phis].sum(), b[betas].sum()
        grad = pd.Series(0.0, index=b.index)
        grad[betas] = 1 / den
        grad[phis] = num / den ** 2
        se = np.sqrt(grad @ V @ grad)
        rows[var] = {"impact β0": b[f"{var}.L0"], "sum of β": num, "1 - sum φ": den,
                     "long run θ": num / den, "s.e.": se, "z": num / den / se}
    return pd.DataFrame(rows).T


print(long_run_multipliers(ardl_res, ARDL_P, ARDL_Q).round(4))

uecm = UECM(insample["y"], ARDL_P, insample[list(ARDL_Q)], ARDL_Q, trend="c").fit()
print(uecm.summary().tables[1])
bounds = uecm.bounds_test(case=3)   # case 3: unrestricted constant, no trend
print(f"Bounds F-stat = {bounds.statistic:.2f} | p-value with I(0) bounds = {bounds.pvalue['lower']:.2g}, I(1) bounds = {bounds.pvalue['upper']:.2g}")
print(f"t-stat of the error-correction term π_y = {uecm.tvalues['y.L1']:.2f} (PSS 5% t-bounds for k=1: -2.86 / -3.22)")

**Orders.**
* **BIC** keeps only unemployment: ARDL$(3; q_u = 1)$, i.e. 6 parameters.
* **AIC** keeps all four covariates with up to 4 lags of $u$ and 3 of $ca$: about 17 parameters on 44 usable observations. Its backtest MASE is **9.4**, against 2.3 for BIC. This is textbook over-fitting and the reason we use BIC.

**Short-run vs long-run multipliers.**
* The impact effect of unemployment is positive, $\beta_{u,0} = +0.0175$ (t = 2.8): a 1 pp rise in $u$ raises the charge-off rate by 1.75 bp in the same quarter.
* It is reversed the next quarter, $\beta_{u,1} = -0.0253$. In practice, what matters is the *change* in unemployment, not its level.
* The long-run multiplier is $\theta_u = -0.0078 / 0.2735 = -0.029$ with a delta-method standard error of 0.023 ($z = -1.2$): no significant long-run effect.

**Long-run relationship.** In the error-correction form, $\pi_y = -0.27$ (t = −3.49): about 27 % of any gap to the long-run level is closed each quarter. The bounds F-statistic is 14.9, far above the I(1) upper bound ($p \approx 5 \times 10^{-7}$), and the t-statistic is beyond the −3.22 bound. Formally, we reject "no level relationship".

**This conclusion needs care.** $\pi_u$ itself is not significant ($p = 0.12$). The rejection is driven by $\pi_y$, i.e. by $y$ reverting to a constant, which we already knew from the ADF test. PSS call this the *degenerate* case: the bounds test can reject only because the dependent variable is I(0). So we do **not** find a cointegrating relation between the charge-off rate and unemployment. We find a mean-reverting rate that reacts to *changes* in unemployment.

**ARDL vs ARIMAX.**
* The ARDL lets each covariate act with its own dynamics: the impact differs from the long run, and the response spreads over time. All persistence goes through the lags of $y$, with white-noise errors (clean here: every Ljung–Box p > 0.3 and normality not rejected).
* The ARIMAX allows MA terms and puts the dynamics in the error. But it forces the effect of $x$ to be immediate and identical in the short and long run (common-factor restriction).

The ARIMAX forecasts better at $h = 1$ (MASE 1.00 vs 2.31), because the MA term uses the last shock. Over horizons 1–8 the two are close (1.86 vs 2.29).

In [ ]:
backtests_aic = {"ARDL (AIC)": ardl_aic_bt}  # kept aside: shown only as a warning against AIC

### 2.5 Darts: one machine-learning and one deep-learning model

**Past vs future covariates.** In Darts, *past covariates* are known only up to the forecast origin (the model can only use their lags ≥ 1), *future covariates* are known over the forecast horizon (the model can use their value at $t$, or even $t+1$). Since in Question 3 we condition every forecast on the WEO path, **the four macro variables are future covariates**. This gives the Darts models exactly the same information as the ARIMAX and the ARDL, so the comparison is fair.

**Models.**
* **`LinearRegressionModel`** (benchmark): OLS of $y_t$ on $y_{t-1}, \dots, y_{t-L}$ and on the covariates at $t$ (and $t-1$). It is a tabular ARDL. With `output_chunk_length=1`, multi-step forecasts are **iterated**: Darts feeds its own forecast back as a lag and reads the future covariates at each step.
* **`LightGBMModel`** (machine learning): gradient-boosted regression trees on the same lagged features. Each new tree is fitted on the current residuals (the gradient of the squared loss) and added with a small learning rate. Trees are piecewise constant, so they **cannot extrapolate** beyond the values seen in training. Regularisation: very small trees (`num_leaves=4`), a minimum number of points per leaf, subsampling of rows and columns.
* **`TiDEModel`** (deep learning, Das et al. 2023): an MLP encoder–decoder. The past of $y$ and the covariates are encoded by small residual MLPs into a vector, decoded into the whole output window, then each horizon is combined with that quarter's future covariates; there is a linear skip connection from the past. We chose it because it accepts **future covariates** (N-BEATS and N-HiTS only accept past covariates).

**Lags, scaling, loss.** Target lags $L \in \{1, \dots, 4\}$, covariate lags $\{0\}$ or $\{-1, 0\}$. Trees and OLS need no scaling; for TiDE a min–max `Scaler` is **fitted on the training window only** at each origin (fitting it on the full sample would leak the future range). Loss: squared error for the three models (Adam optimiser with weight decay for TiDE).

**Tuning.** Hyper-parameters are chosen on the same expanding-window backtest as the econometric models. This is exactly what `historical_forecasts(retrain=True)` does in Darts; we use our own `backtest` loop so that all models see the same origins, and we check below that it gives the same numbers. Random seeds are fixed through `random_state`; for TiDE we average 3 seeds.

**A deep network with 48 points?** Not a reasonable first choice: with a look-back of 4 quarters, the training window at the first origin contains about 20 samples. We limit over-fitting with a tiny network (hidden size 8–16), dropout, weight decay, few epochs and seed averaging, and we judge it against the naive and ARIMA benchmarks.

**Forecasts and uncertainty.** LR and LightGBM forecast one step and are rolled out (iterated); TiDE forecasts a block of 4 quarters at once (direct) and is called twice for 8 quarters. None of the three gives a reliable interval with so little data, so we build **conformal** intervals: at each horizon, the 80 % quantile of the absolute errors of the in-sample backtest.
**Limits.** Very few training samples (the trees see about 40 rows at the last origin), one big outlier (2020), no extrapolation for trees, results that depend on the random seed for TiDE, and forecasts that are only as good as the WEO path we feed in.

In [ ]:
import logging
import lightgbm  # load LightGBM's OpenMP before torch's, otherwise the kernel segfaults on macOS
import torch
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import LinearRegressionModel, LightGBMModel, TiDEModel

for name in ["pytorch_lightning", "lightning.pytorch", "darts"]:
    logging.getLogger(name).setLevel(logging.ERROR)
torch.manual_seed(SEED)


def to_ts(df):
    """pandas (PeriodIndex) -> Darts TimeSeries."""
    df = df.to_frame() if isinstance(df, pd.Series) else df.copy()
    df.index = df.index.to_timestamp()
    return TimeSeries.from_dataframe(df, freq="QS")


def darts_model(build, cols=X, scale=False, seeds=(SEED,)):
    """Wrap a Darts model into our f(train, x_future, h) interface; covariates = future covariates."""
    cols = list(cols)
    def f(train, x_future, h):
        y = to_ts(train["y"])
        cov = to_ts(pd.concat([train[cols], x_future[cols]])) if cols else None
        if scale:  # scalers fitted on the training window only
            y_scaler = Scaler()
            y = y_scaler.fit_transform(y)
            if cols:
                cov = Scaler().fit(to_ts(train[cols])).transform(cov)
        forecasts = []
        for s in seeds:
            model = build(s)
            model.fit(y, future_covariates=cov)
            pred = model.predict(h, series=y, future_covariates=cov)
            if scale:
                pred = y_scaler.inverse_transform(pred)
            forecasts.append(pred.values().ravel())
        nan = np.full(h, np.nan)   # intervals are added later (conformal, section 3)
        return output(np.mean(forecasts, axis=0), nan, nan, x_future.index)
    return f


def linreg(L, fut_lags):
    return lambda s: LinearRegressionModel(lags=int(L), lags_future_covariates=fut_lags, output_chunk_length=1, random_state=s)


def lightgbm(L, fut_lags, min_leaf=5, n_trees=200):
    return lambda s: LightGBMModel(lags=int(L), lags_future_covariates=fut_lags, output_chunk_length=1, random_state=s,
                                   num_leaves=4, min_child_samples=int(min_leaf), n_estimators=int(n_trees), learning_rate=0.05,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)


def tide(icl=4, hidden=8, dropout=0.3):
    return lambda s: TiDEModel(input_chunk_length=icl, output_chunk_length=4, hidden_size=hidden,
                               num_encoder_layers=1, num_decoder_layers=1, decoder_output_dim=4,
                               temporal_decoder_hidden=8, dropout=dropout, n_epochs=100, batch_size=16,
                               random_state=s, optimizer_kwargs={"lr": 3e-3, "weight_decay": 1e-3},
                               pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": False,
                                                  "enable_model_summary": False, "logger": False})

In [ ]:
# Linear regression: grid on the number of target lags, the covariate lags and the covariate set
rows = []
for L, fl, cols in itertools.product([1, 2, 3, 4], [[0], [-1, 0]], [X, ["u"], ["g", "u"]]):
    bt = backtest(darts_model(linreg(L, fl), cols))
    rows.append({"L": L, "cov lags": fl, "covariates": "+".join(cols), "MASE": scores(bt)["MASE"], "bt": bt})
lr_grid = pd.DataFrame(rows).sort_values("MASE")
LR_BEST = lr_grid.iloc[0]
backtests["Darts-LR"] = LR_BEST["bt"]
lr_grid.drop(columns="bt").head(6).round(3)

In [ ]:
# LightGBM: grid on lags and regularisation (all four covariates, trees select them)
rows = []
for L, fl, min_leaf, n_trees in itertools.product([2, 4], [[0], [-1, 0]], [3, 6], [100, 300]):
    bt = backtest(darts_model(lightgbm(L, fl, min_leaf, n_trees)))
    rows.append({"L": L, "cov lags": fl, "min_leaf": min_leaf, "trees": n_trees, "MASE": scores(bt)["MASE"], "bt": bt})
lgb_grid = pd.DataFrame(rows).sort_values("MASE")
LGB_BEST = lgb_grid.iloc[0]
backtests["Darts-LightGBM"] = LGB_BEST["bt"]
lgb_grid.drop(columns="bt").head(6).round(3)

In [ ]:
# check: our loop gives the same 1-step forecasts as Darts' historical_forecasts
best_lgb = lightgbm(LGB_BEST.L, LGB_BEST["cov lags"], LGB_BEST.min_leaf, LGB_BEST.trees)(SEED)
hf = best_lgb.historical_forecasts(to_ts(insample["y"]), future_covariates=to_ts(insample[X]),
                                   start=pd.Timestamp("2019-01-01"), forecast_horizon=1, retrain=True, verbose=False)
ours = backtests["Darts-LightGBM"].query("h == 1").set_index("target")["mean"]
hf = pd.Series(hf.values().ravel(), index=pd.PeriodIndex(hf.time_index, freq="Q"))
print("max difference with historical_forecasts:", float((ours.reindex(hf.index) - hf).abs().max()))

In [ ]:
# TiDE: small grid screened on every other origin with one seed (to save time), then 3 seeds on all origins
rows = []
for icl, hidden, dropout in itertools.product([4, 8], [8, 16], [0.1, 0.3]):
    bt = backtest(darts_model(tide(icl, hidden, dropout), scale=True), origins=BT_ORIGINS[::2])
    rows.append({"look-back": icl, "hidden": hidden, "dropout": dropout, "MASE": scores(bt)["MASE"]})
tide_grid = pd.DataFrame(rows).sort_values("MASE")
TIDE_BEST = tide_grid.iloc[0]
TIDE_SEEDS = (SEED, SEED + 1, SEED + 2)
tide_best = tide(int(TIDE_BEST["look-back"]), int(TIDE_BEST.hidden), TIDE_BEST.dropout)
backtests["Darts-TiDE"] = backtest(darts_model(tide_best, scale=True, seeds=TIDE_SEEDS))
print("Seed-averaged MASE:", round(scores(backtests["Darts-TiDE"])["MASE"], 3))
tide_grid.round(3)

**Linear regression.** The best specification is the simplest one: one lag of $y$ and contemporaneous $g$ and $u$, i.e. an ARX(1). Its backtest MASE is **1.73**, the best of all models so far. Adding inflation and the current account, or more lags, always makes it worse.

**LightGBM.** The best setting uses 4 lags of $y$, covariates at $t$ and $t-1$, at least 3 points per leaf and 300 trees, for a MASE of 2.27. That is worse than the linear model. With 20–40 rows per training window, trees can only learn a few step functions, and they cannot extrapolate beyond the range seen in training.

**Check against Darts.** Our backtest loop and `historical_forecasts(retrain=True)` give *identical* one-step forecasts (maximum difference 0.0). So the tuning protocol is exactly the Darts one.

**TiDE.**
* The screening picks a look-back of 8 quarters, hidden size 16 and dropout 0.3. Averaged over 3 seeds, the full backtest MASE is 2.21.
* Its profile is unusual: poor at $h = 1$ (2.14), but the best of all models at $h = 8$ (1.61). Because it forecasts directly, it learns the average mean reversion over a year instead of rolling out one-step errors.
* The grid ranges from 2.35 to 3.12, depending only on architecture choices that have no economic meaning. Hyperparameter noise is as large as the gaps between models, which is the main warning about deep learning on 48 points.

**Bottom line.** With 48 observations, extra complexity does not pay. The best Darts model is a linear regression that an econometrician would have written by hand.

### 2.6 TimesFM 2.5 (zero-shot foundation model)
**(a) What it is.** A decoder-only transformer (200 M parameters) pretrained by Google on a large corpus of time series. The context is cut into **patches** of 32 points; each patch is embedded by a small MLP; causal self-attention runs over the patches; each output token is decoded into a patch of 128 future points, so our 8-quarter horizon needs a single forward pass. **Instance normalisation**: each context is standardised by its own mean and standard deviation, and the forecast is de-normalised, so the model is scale-free. **Context length** up to 16k points (our 48 quarters fill 1.5 patches, the rest is padding). A **quantile head** gives the 10 %, …, 90 % quantiles, which we use as the 80 % interval.
**(b) Estimation.** None on our data: the weights are fixed, the forecast is **zero-shot**.
**(c) Hyper-parameters.** The inference options (`max_context`, normalisation, positivity of the output) are fixed a priori. For XReg, the covariate set and the ridge penalty of the regression are chosen on the same backtest as the other models.
**(d) Covariates.** The transformer itself never sees the covariates. `forecast_with_covariates` (XReg) fits a **linear (ridge) regression of $y$ on the covariates in the context window** and combines it with TimesFM: in mode `"xreg + timesfm"` the regression is fitted first and TimesFM forecasts its residuals; in `"timesfm + xreg"` TimesFM forecasts first and the regression explains its errors. We can only use the first mode: the second one fits the regression on TimesFM's in-context errors, which exist only after the first patch of 32 points, so with 28 to 44 quarters of history it has 0 to 12 points to work with (and the library crashes below 32). Over the horizon the covariates are the WEO path, as for the ARIMAX.
**(e) Uncertainty.** Quantile heads (with XReg the regression part is added to every quantile, so the uncertainty on the regression coefficients is ignored).
**(f) Limits.** Not designed for 48-point macro series, opaque, linear and static use of covariates, and possible contamination of the pretraining data (discussed below).

**What is an "in-sample fit" for a model that is never estimated on our data?** It has no meaning: there are no estimated parameters and no residuals. What we *can* do is put TimesFM through exactly the same rolling backtest as the other models: at each origin it receives the data up to that origin and forecasts the next quarters. That is a fair in-sample comparison.

**Could TimesFM have seen our data?** The model card describes the pretraining corpus only by sources (Google Trends, Wikipedia page views, synthetic series and public benchmark archives such as M4 and the LOTSA collection). FRED series are widely redistributed in such archives, so we **cannot rule out** that CORBLACBS or its covariates were in the training set. TimesFM 2.5 was released in September 2025, after the end of 2024, so data covering 2024 and part of 2025 may also have been seen. If so, its out-of-sample "forecasts" of 2024–2025 could partly be memory, and a good TimesFM score there should be discounted.

In [ ]:
import timesfm

TFM_CONFIG = timesfm.ForecastConfig(max_context=64, max_horizon=128, normalize_inputs=True,
                                    use_continuous_quantile_head=True, force_flip_invariance=True,
                                    infer_is_positive=True, fix_quantile_crossing=True,
                                    return_backcast=True)  # return_backcast is required by forecast_with_covariates

try:
    tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=False)
    # with old torch versions (2.2 on Intel Macs) the fused attention returns NaN on padded inputs:
    # we switch every layer to TimesFM's own plain attention, which computes exactly the same thing
    from timesfm.torch import transformer as tfm_transformer
    for layer in tfm.model.modules():
        if hasattr(layer, "attention_fn"):
            layer.attention_fn = tfm_transformer._dot_product_attention
    tfm.compile(TFM_CONFIG)
    TFM_OK = True
except Exception as err:
    print("TimesFM checkpoint not available:", type(err).__name__)
    TFM_OK = False


def tfm_output(point, quantiles, index):
    # with return_backcast=True, forecast() puts the backcast first: the forecast is the LAST h values
    # quantiles[:, 0] is the mean, then the 10%, 20%, ..., 90% quantiles
    h = len(index)
    point, quantiles = np.asarray(point)[-h:], np.asarray(quantiles)[-h:]
    return output(point, quantiles[:, 1], quantiles[:, 9], index)


def timesfm_zero_shot(train, x_future, h):
    point, quantiles = tfm.forecast(horizon=h, inputs=[train["y"].values])
    return tfm_output(point[0], quantiles[0], x_future.index)


def timesfm_xreg(cols=X, ridge=1.0):
    cols = list(cols)
    def f(train, x_future, h):
        # each covariate must cover the context AND the horizon
        covariates = {c: [np.r_[train[c].values, x_future[c].values]] for c in cols}
        point, quantiles = tfm.forecast_with_covariates(inputs=[train["y"].values], dynamic_numerical_covariates=covariates,
                                                        xreg_mode="xreg + timesfm", ridge=ridge, force_on_cpu=True)
        return tfm_output(point[0], quantiles[0], x_future.index)
    return f


if TFM_OK:
    backtests["TimesFM zero-shot"] = backtest(timesfm_zero_shot)
    rows = []
    for ridge, cols in itertools.product([0.0, 1.0], [X, ["u"], ["g", "u"]]):
        bt = backtest(timesfm_xreg(cols, ridge))
        rows.append({"ridge": ridge, "covariates": "+".join(cols), **scores(bt)[["MASE", "RMSE", "Coverage 80%"]], "bt": bt, "cols": cols})
    tfm_grid = pd.DataFrame(rows).sort_values("MASE")
    TFM_BEST = tfm_grid.iloc[0]
    backtests["TimesFM-XReg"] = TFM_BEST["bt"]
    print("Zero-shot:", scores(backtests["TimesFM zero-shot"]).round(3).to_dict())
    display(tfm_grid.drop(columns=["bt", "cols"]).round(3))

ANALYSIS_TIMESFM_PENDING

### 2.7 In-sample comparison
All the models went through the same backtest (17 origins, horizons 1 to 8, targets ≤ 2023Q4, 126 errors per model). We also test whether the differences are significant with the **Diebold–Mariano** test (Harvey–Leybourne–Newbold small-sample correction) at horizons 1, 4 and 8.

*Caveat.* The orders were chosen on the whole in-sample period and then backtested inside it, so the backtest is slightly optimistic — for all models in the same way.

In [ ]:
def diebold_mariano(e1, e2, h=1):
    """DM test on squared errors with the HLN correction. Negative stat = model 1 better."""
    d = (e1 ** 2 - e2 ** 2).dropna().values
    T = len(d)
    gamma = [np.mean((d[k:] - d.mean()) * (d[:T - k] - d.mean())) for k in range(h)]
    long_run_var = max(gamma[0] + 2 * sum(gamma[1:]), gamma[0])
    stat = d.mean() / np.sqrt(long_run_var / T) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * stats.t.sf(abs(stat), df=T - 1)


comparison = pd.DataFrame({m: scores(bt) for m, bt in backtests.items()}).T
comparison["Theil U"] = comparison["RMSE"] / comparison.loc["Naive (RW)", "RMSE"]
by_h = pd.DataFrame({m: mase_by_h(bt) for m, bt in backtests.items()}).T.add_prefix("h=")
comparison = pd.concat([comparison, by_h[["h=1", "h=2", "h=4", "h=8"]]], axis=1).sort_values("MASE")
IS_RANK = comparison["MASE"].rank()
comparison.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for m in comparison.index:
    ax.plot(range(1, 9), mase_by_h(backtests[m]).values, "o-", ms=3, lw=2.5 if m in comparison.index[:3] else 1, label=m)
ax.set_xlabel("horizon (quarters)")
ax.set_ylabel("MASE")
ax.set_title("In-sample backtest: MASE by horizon")
ax.legend(fontsize=7, ncol=3)
plt.show()

best = comparison.index[0]
rows = {}
for m in comparison.index:
    for ref in ["Naive (RW)", best]:
        if m == ref:
            continue
        for h in (1, 4, 8):
            e1 = backtests[m].query("h == @h").set_index("target")["error"]
            e2 = backtests[ref].query("h == @h").set_index("target")["error"]
            rows.setdefault(m, {})[(f"vs {ref}", f"h={h}")] = diebold_mariano(e1, e2.reindex(e1.index), h)[1]
print("Diebold-Mariano p-values:")
pd.DataFrame(rows).T.round(3)

**Best in-sample model: Darts-LR** (MASE 1.73, Theil's U 0.53). It is followed by the ARIMAX (1.86) and the ARIMA (2.16). All models beat the random walk (MASE 3.49); the RW is ranked last together with ARIMA(0,1,0), which is the same model and gives identical point forecasts (a useful sanity check).

**By horizon.** The ARIMAX is best at $h = 1$ (1.00), the linear Darts model at $h = 2$–$4$, and TiDE at $h = 8$. No model dominates at every horizon.

**Is the ranking significant? Mostly not.**
* Against Darts-LR, no model is significantly worse at $h = 4$ or $h = 8$. The only rejection is TiDE at $h = 1$ ($p = 0.03$).
* Against the random walk, only the ARIMA and TiDE are significantly better, and only at $h = 4$.

The 126 backtest errors come from 17 origins in a row and overlap heavily, so the effective sample is small. The differences between the top five models are within noise.

**Intervals.** The Gaussian 80 % intervals of the econometric models cover only 49–60 % of the outcomes. They ignore parameter uncertainty and the fat tails seen in the residuals.

**Caveat.** The orders were chosen on the whole 2012–2023 sample and then backtested inside it, which flatters every model a bit. The out-of-sample test of Question 4 is the real exam.

## 3. Out-of-sample forecasts 2024Q1–2025Q4 (origin 2023Q4)
### 3.1 The scenario
The WEO October 2023 scenario was built in section 1.2: annual values attached to the middle of each year (annual averages), cubic spline connected to the last observed quarter 2023Q4, knots adjusted so that the four quarters average **exactly** to the WEO figure (checked in the table of section 1.2), and WEO growth rates chained on FRED levels to avoid any base-year or vintage mismatch.

### 3.2 How each model produces its 8 forecasts

| Model | Multi-step method | Covariates over the horizon | 80 % interval |
|---|---|---|---|
| Naive | $\hat y_{T+h} = y_T$ | — | $\pm z\,\sigma_{\Delta y}\sqrt h$ |
| ARIMA | iterated (Kalman filter) | — | Gaussian, $\sigma^2\sum_{j<h}\psi_j^2$ |
| ARIMAX | iterated for the ARMA error, $+\hat\beta' x_{T+h}$ | WEO path | Gaussian, conditional on the scenario |
| ARDL | dynamic (iterated) on $\hat y$ | WEO path (contemporaneous and lags) | Gaussian ($\psi$-weights), conditional |
| Darts LR / LightGBM | iterated: the forecast is fed back as a lag | WEO path (future covariates) | conformal: $\pm$ 80 % quantile of the absolute backtest errors at the same horizon |
| Darts TiDE | direct by chunks of 4 (2 passes for $h = 8$), 3 seeds averaged | WEO path | conformal, as above |
| TimesFM | direct: one forward pass gives the 8 points | XReg: in-context linear regression on the WEO path | quantile heads 10 % / 90 % |

All the conditional forecasts depend on the WEO scenario, and their intervals **ignore the uncertainty of the scenario itself**. Question 4.2 measures it.

We also compute **ex-post** forecasts: the same models, estimated on the same data, but with the realised covariates instead of the WEO path (unknown in 2023Q4, used only for diagnosis).

In [ ]:
OOS = pd.period_range(IS_END + 1, OOS_END, freq="Q")
y_oos = panel.loc[OOS, "y"]

MODELS = {
    "Naive (RW)": naive,
    f"ARIMA{ARIMA_ORDER}": arima_model(ARIMA_ORDER),
    f"ARIMAX[{'+'.join(ARIMAX_COLS)}]": arima_model(ARIMA_ORDER, ARIMAX_COLS),
    "ARDL (BIC)": ardl_model(ARDL_P, ARDL_Q),
    "Darts-LR": darts_model(linreg(LR_BEST.L, LR_BEST["cov lags"]), LR_BEST.covariates.split("+")),
    "Darts-LightGBM": darts_model(lightgbm(LGB_BEST.L, LGB_BEST["cov lags"], LGB_BEST.min_leaf, LGB_BEST.trees)),
    "Darts-TiDE": darts_model(tide_best, scale=True, seeds=TIDE_SEEDS),
}
if TFM_OK:
    MODELS["TimesFM zero-shot"] = timesfm_zero_shot
    MODELS["TimesFM-XReg"] = timesfm_xreg(TFM_BEST["cols"], TFM_BEST["ridge"])


def conformal_band(bt):
    """80% quantile of the absolute backtest errors, horizon by horizon."""
    return bt.groupby("h")["error"].apply(lambda e: np.quantile(e.abs(), 1 - ALPHA))


def forecast_all(models, train, x_future, h, bts):
    out = {}
    for name, f in models.items():
        fc = f(train, x_future, h)
        if name.startswith("Darts"):
            band = conformal_band(bts[name]).reindex(range(1, h + 1)).ffill().values
            fc["lo"], fc["hi"] = fc["mean"] - band, fc["mean"] + band
        out[name] = fc
    return out


fc_ante = forecast_all(MODELS, insample, scen23, H, backtests)                  # WEO scenario
fc_post = forecast_all(MODELS, insample, panel.loc[OOS, X], H, backtests)      # realised covariates

table = pd.DataFrame({m: f["mean"] for m, f in fc_ante.items()})
table.insert(0, "Actual", y_oos)
table.round(3)

In [ ]:
n = len(fc_ante)
fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(14, 3 * np.ceil(n / 3)), sharey=True)
history = panel.loc["2019Q1":IS_END, "y"]
for ax, (m, f) in zip(axes.flat, fc_ante.items()):
    ax.plot(history.index.to_timestamp(), history.values, color="k")
    ax.plot(OOS.to_timestamp(), y_oos.values, "ko--", ms=3, label="actual")
    ax.plot(OOS.to_timestamp(), f["mean"].values, "o-", ms=3, color="tab:blue", label="forecast (WEO)")
    ax.fill_between(OOS.to_timestamp(), f["lo"].values, f["hi"].values, color="tab:blue", alpha=0.2, label="80% interval")
    ax.plot(OOS.to_timestamp(), fc_post[m]["mean"].values, ":", color="tab:green", label="ex post (realised x)")
    ax.set_title(m, fontsize=9)
for ax in axes.flat[n:]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

**What happened.** The charge-off rate kept rising, from 0.44 % in 2024Q1 to 0.58 % in 2025Q2, and then stayed there.

**What the models predicted.**
* Almost every model predicts a turning point and a fall back towards the 2012–2023 mean of about 0.3 %. The ARIMA follows its 18-quarter cycle: a peak at 0.51 % in 2024Q3, then 0.31 % in 2025Q4. The ARIMAX ends at 0.24 %, the ARDL at 0.35 % and TiDE at 0.27 %.
* The reason is simple: in-sample, every high was followed by a fall, so the models learned strong mean reversion. 2024–2025 looks like a new regime (higher interest rates, tighter credit), in which the rate stays high.
* Only Darts-LR (around 0.46–0.48 %) and the random walk (0.42 %) stay reasonably close.

**Ex post (green dotted line).** For Darts-LR and LightGBM, the forecasts with the *realised* covariates are *lower* than with the WEO path. The real economy grew faster than the WEO expected, and these models read strong growth as fewer defaults. Section 4.2 measures this.

**Intervals.** The random walk and Darts-LR intervals contain all 8 outcomes. The ARIMAX and ARDL intervals miss 6 of the 8: they are both too narrow and centred too low.

## 4. Forecast evaluation
### 4.1 Accuracy over 2024Q1–2025Q4
Same metrics as in 2.7 (MASE with the in-sample scale, RMSE, Theil's U, bias, coverage of the 80 % intervals).

In [ ]:
def oos_backtest(fcs):
    """Put the 8 out-of-sample forecasts of each model in the same format as a backtest."""
    out = {}
    for m, f in fcs.items():
        d = f.copy()
        d["y"] = y_oos
        d["error"] = d["y"] - d["mean"]
        d["h"] = range(1, len(d) + 1)
        out[m] = d
    return out


oos_ante, oos_post = oos_backtest(fc_ante), oos_backtest(fc_post)
evaluation = pd.DataFrame({m: scores(d) for m, d in oos_ante.items()}).T
evaluation["Theil U"] = evaluation["RMSE"] / evaluation.loc["Naive (RW)", "RMSE"]
evaluation["MASE ex post"] = [scores(oos_post[m])["MASE"] for m in evaluation.index]
evaluation["in-sample rank"] = IS_RANK.reindex(evaluation.index).rank()
evaluation["out-of-sample rank"] = evaluation["MASE"].rank()
evaluation = evaluation.sort_values("MASE")
rho = stats.spearmanr(evaluation["in-sample rank"], evaluation["out-of-sample rank"])
print(f"Spearman correlation between the in-sample and out-of-sample ranks: {rho.statistic:.2f} (p = {rho.pvalue:.2f})")
evaluation.round(3)

**Winner: Darts-LR** (MASE 1.56, Theil's U 0.68). It is the only model clearly better than the random walk (MASE 2.24). The ARIMA comes second (2.06).

**Biases.** Every model has a positive bias of 0.08–0.20 pp, i.e. they all under-predict. This is the mean-reversion problem described above.

**Is the in-sample ranking preserved?** Only at the top. The Spearman rank correlation is 0.21 ($p = 0.64$). The ARIMAX falls from 2nd to 6th (MASE 3.51) and TiDE from 4th to last (3.96). Meanwhile the random walk, last in-sample, comes 3rd.

**How much can 8 observations say?** Very little. They come from a single origin, all models share the same scenario, and the errors are highly autocorrelated (once a model is below the path, it stays below). In practice, this is closer to *one* draw than to eight independent tests. A Diebold–Mariano test on 8 overlapping multi-step errors would have no power, so we do not report one. The next section also shows that part of Darts-LR's win is luck.

### 4.2 Model error vs scenario error
For each model the ex-ante error splits exactly into two parts:
$$\underbrace{y_{T+h} - \hat y_{T+h}(x^{WEO})}_{\text{total error}} = \underbrace{y_{T+h} - \hat y_{T+h}(x^{real})}_{\text{model error}} + \underbrace{\hat y_{T+h}(x^{real}) - \hat y_{T+h}(x^{WEO})}_{\text{scenario error}}$$

In [ ]:
annual = pd.DataFrame(index=[2024, 2025])
for y in annual.index:
    annual.loc[y, "GDP growth: WEO"] = weo23.loc[y, "NGDP_RPCH"]
    annual.loc[y, "GDP growth: realised"] = 100 * (raw.loc[str(y), "GDPC1"].mean() / raw.loc[str(y - 1), "GDPC1"].mean() - 1)
    annual.loc[y, "inflation: WEO"] = weo23.loc[y, "PCPIPCH"]
    annual.loc[y, "inflation: realised"] = 100 * (raw.loc[str(y), "CPIAUCSL"].mean() / raw.loc[str(y - 1), "CPIAUCSL"].mean() - 1)
    annual.loc[y, "unemployment: WEO"] = weo23.loc[y, "LUR"]
    annual.loc[y, "unemployment: realised"] = panel.loc[str(y), "u"].mean()
    annual.loc[y, "CA/GDP: WEO"] = weo23.loc[y, "BCA_NGDPD"]
    annual.loc[y, "CA/GDP: realised"] = panel.loc[str(y), "ca"].mean()
print(annual.T.round(2))

rows = {}
for m in fc_ante:
    total, model_err = oos_ante[m]["error"], oos_post[m]["error"]
    scenario_err = fc_post[m]["mean"] - fc_ante[m]["mean"]
    rows[m] = {"RMSE ex ante": np.sqrt((total ** 2).mean()), "RMSE ex post": np.sqrt((model_err ** 2).mean()),
               "mean total error": total.mean(), "mean model error": model_err.mean(), "mean scenario error": scenario_err.mean()}
pd.DataFrame(rows).T.sort_values("RMSE ex ante").round(4)

**WEO vs reality (annual).**
* Growth was 2.95 % / 2.34 % in 2024/2025, against 1.5 % / 1.8 % in the WEO.
* Unemployment was 4.0 % / 4.3 %, against 3.85 % / 3.9 %.
* Inflation was 0.2–0.3 pp above the WEO.
* The current-account deficit was 1.2–1.3 pp of GDP wider.

**Model error vs scenario error.**
* **Darts-LR** has an ex-ante RMSE of 0.082, but **0.146 ex post**. Its model error (+0.136 on average) is partly cancelled by a scenario error of −0.059. The WEO's pessimistic growth path pushed its forecasts *up*, by chance in the right direction. With the true covariates its MASE would be 2.74, worse than the random walk. **Its out-of-sample win is mainly two errors cancelling out.**
* **LightGBM** tells the same story (ex-post MASE 3.48).
* For the **ARIMAX and ARDL**, the scenario error is almost zero, because their unemployment coefficient is tiny. Their whole error is model error.

**Conclusion.** The forecast failure of 2024–2025 is a *model* failure, not a scenario failure: no model expected the rate to stay high. Once the scenario luck is removed, only the univariate ARIMA beats the random walk (2.06 vs 2.24), and the covariates do not help.

### 4.3 Forecast combinations
The weights are computed **on the in-sample backtest only**: simple average, median, and inverse-MSE weights $w_m \propto 1/\text{MSE}_m$ (Bates–Granger without covariances, too noisy to estimate here). The naive forecast is not in the pool.

In [ ]:
pool = [m for m in fc_ante if m != "Naive (RW)"]
mse = pd.Series({m: (backtests[m]["error"] ** 2).mean() for m in pool})
weights = (1 / mse) / (1 / mse).sum()
print("Inverse-MSE weights:", weights.round(3).to_dict())

ante = pd.DataFrame({m: fc_ante[m]["mean"] for m in pool})
post = pd.DataFrame({m: fc_post[m]["mean"] for m in pool})
bt_means = pd.DataFrame({m: backtests[m].set_index(["origin", "target"])["mean"] for m in pool})
bt_y = backtests["Naive (RW)"].set_index(["origin", "target"])["y"].reindex(bt_means.index)

combos = {"Mean": lambda F: F.mean(axis=1), "Median": lambda F: F.median(axis=1), "Inverse-MSE": lambda F: F @ weights}
rows = {}
for name, comb in combos.items():
    rows[name] = {"backtest MASE": (bt_y - comb(bt_means)).abs().mean() / MASE_SCALE,
                  "OOS MASE ex ante": (y_oos - comb(ante)).abs().mean() / MASE_SCALE,
                  "OOS MASE ex post": (y_oos - comb(post)).abs().mean() / MASE_SCALE}
for m in evaluation.index[:3]:
    rows[m] = {"backtest MASE": comparison.loc[m, "MASE"], "OOS MASE ex ante": evaluation.loc[m, "MASE"],
               "OOS MASE ex post": evaluation.loc[m, "MASE ex post"]}
pd.DataFrame(rows).T.round(3)

**The weights are fairly even**, from 0.07 (ARDL) to 0.23 (Darts-LR): no model is much more accurate than the others in the backtest.

**In-sample**, the combinations are better than every single model: MASE 1.41 for the mean vs 1.73 for the best model. Models of different nature make different errors, and averaging cancels part of them.

**Out of sample**, it goes the other way. The combinations score MASE 2.46–2.58, worse than Darts-LR, the ARIMA and even the random walk. A combination reduces the *variance* of the errors, not a *common bias*, and in 2024–2025 all models were biased the same way. The combination simply averages the shared under-prediction. The simple mean and the inverse-MSE weights give almost the same result, the usual "forecast combination puzzle".

## 5. Update: forecast of 2026Q3
The sample now ends in **2026Q2** (last published charge-off rate at the forecast origin) and the scenario is the **WEO April 2026**, the last vintage published before the end of 2026Q2, so no later information is used. We **re-run the selection** with the same rules. Since the target is now one quarter ahead, the backtest criterion is the MASE at $h = 1$ (origins 2018Q4 to 2025Q2).

In [ ]:
upd = panel.loc[IS_START:UPD_END]
weo26 = weo[weo.vintage == "2026-04"].set_index("year")
scen26, _ = weo_scenario(raw, weo26, UPD_END, pd.Period("2027Q2"))
print("WEO April 2026 scenario:\n", scen26.round(3))
UPD_ORIGINS = pd.period_range("2018Q4", "2025Q2", freq="Q")


def backtest_h1(model):
    # models are run up to h=4 (TiDE needs a full output chunk) but only h=1 is scored
    return backtest(model, origins=UPD_ORIGINS, h=4, end=UPD_END).query("h == 1")


# ARIMA: BIC short list then h=1 backtest
rows = []
for d, p, q in itertools.product([0, 1], range(4), range(4)):
    rows.append({"order": (p, d, q), "d": d, "BIC": fit_arima(upd["y"], (p, d, q)).bic})
upd_arima = pd.concat([pd.DataFrame(rows).query("d == @d").nsmallest(2, "BIC") for d in (0, 1)])
upd_arima["MASE h=1"] = [scores(backtest_h1(arima_model(o)))["MASE"] for o in upd_arima["order"]]
ARIMA_U = upd_arima.sort_values("MASE h=1")["order"].iloc[0]

# ARIMAX: best BIC among the covariate subsets
rows = []
for k in range(1, 5):
    for cols in itertools.combinations(X, k):
        rows.append({"cols": list(cols), "BIC": fit_arima(upd["y"], ARIMA_U, exog=upd[list(cols)]).bic})
ARIMAX_U = pd.DataFrame(rows).sort_values("BIC")["cols"].iloc[0]

# ARDL: BIC
sel_u = ardl_select_order(upd["y"], 4, upd[X], 4, ic="bic", trend="c")
ARDL_U = (max(sel_u.model.ar_lags), {v: max(l) for v, l in sel_u.model.dl_lags.items()})

# Darts LR and LightGBM: same grids, h=1 criterion
rows = []
for L, fl, cols in itertools.product([1, 2, 3, 4], [[0], [-1, 0]], [X, ["u"], ["g", "u"]]):
    rows.append({"L": L, "fl": fl, "cols": cols, "MASE": scores(backtest_h1(darts_model(linreg(L, fl), cols)))["MASE"]})
LR_U = pd.DataFrame(rows).sort_values("MASE").iloc[0]
rows = []
for L, fl, ml, nt in itertools.product([2, 4], [[0], [-1, 0]], [3, 6], [100, 300]):
    rows.append({"L": L, "fl": fl, "min_leaf": ml, "trees": nt, "MASE": scores(backtest_h1(darts_model(lightgbm(L, fl, ml, nt))))["MASE"]})
LGB_U = pd.DataFrame(rows).sort_values("MASE").iloc[0]

pd.DataFrame({
    "2012-2023": [ARIMA_ORDER, ARIMAX_COLS, (ARDL_P, ARDL_Q), (LR_BEST.L, LR_BEST["cov lags"], LR_BEST.covariates),
                  (LGB_BEST.L, LGB_BEST["cov lags"], LGB_BEST.min_leaf, LGB_BEST.trees)],
    "2012-2026Q2": [ARIMA_U, ARIMAX_U, ARDL_U, (LR_U.L, LR_U.fl, "+".join(LR_U.cols)), (LGB_U.L, LGB_U.fl, LGB_U.min_leaf, LGB_U.trees)],
}, index=["ARIMA order", "ARIMAX covariates", "ARDL (p, q)", "Darts-LR (L, lags, x)", "LightGBM (L, lags, leaf, trees)"])

In [ ]:
MODELS_U = {
    "Naive (RW)": naive,
    f"ARIMA{ARIMA_U}": arima_model(ARIMA_U),
    f"ARIMAX[{'+'.join(ARIMAX_U)}]": arima_model(ARIMA_U, ARIMAX_U),
    "ARDL (BIC)": ardl_model(*ARDL_U),
    "Darts-LR": darts_model(linreg(LR_U.L, LR_U.fl), LR_U.cols),
    "Darts-LightGBM": darts_model(lightgbm(LGB_U.L, LGB_U.fl, LGB_U.min_leaf, LGB_U.trees)),
    "Darts-TiDE": darts_model(tide_best, scale=True, seeds=TIDE_SEEDS),
}
if TFM_OK:
    MODELS_U["TimesFM zero-shot"] = timesfm_zero_shot
    MODELS_U["TimesFM-XReg"] = timesfm_xreg(TFM_BEST["cols"], TFM_BEST["ridge"])

bt_upd = {m: backtest_h1(f) for m, f in MODELS_U.items()}
fc_upd = forecast_all(MODELS_U, upd, scen26.iloc[:4], 4, bt_upd)

final = pd.DataFrame({m: {"2026Q3 forecast": f["mean"].iloc[0], "80% low": f["lo"].iloc[0], "80% high": f["hi"].iloc[0],
                          "backtest MASE h=1": scores(bt_upd[m])["MASE"]} for m, f in fc_upd.items()}).T
mse_u = pd.Series({m: (bt_upd[m]["error"] ** 2).mean() for m in final.index if m != "Naive (RW)"})
w_u = (1 / mse_u) / (1 / mse_u).sum()
final.loc["Combination (inverse MSE)", "2026Q3 forecast"] = final.loc[w_u.index, "2026Q3 forecast"] @ w_u
final = final.sort_values("backtest MASE h=1")
final.round(3)

In [ ]:
actual_q3 = panel.loc[UPD_TARGET, "y"] if UPD_TARGET in panel.index else np.nan
if pd.notna(actual_q3):
    print(f"Published 2026Q3 charge-off rate: {actual_q3:.2f} %")
    print((actual_q3 - final["2026Q3 forecast"]).rename("error (actual - forecast)").round(3).to_string())
else:
    print("2026Q3 not published yet on FRED: re-run this cell after the release.")

**What changes with 10 more quarters.**
* The ARIMA becomes **(0,1,2)**. With 2024–2025 the rate no longer falls back after its peak, so a differenced model now wins at $h = 1$.
* The ARIMAX (still $u$) and the ARDL$(3; u{:}1)$ do not change.
* Darts-LR switches to 4 lags of $y$ with $u$ only, and LightGBM to 2 lags with bigger leaves.

**The WEO April 2026 scenario** for 2026Q3: growth 1.9 %, inflation 3.4 %, unemployment 4.3 %, current account −4.2 % of GDP.

**2026Q3 forecasts.**
* Most models are between **0.56 % and 0.60 %**; TiDE is the outlier at 0.49 %.
* At $h = 1$ the best backtest model, the ARIMA(0,1,2) (MASE 1.02), is barely better than the random walk (1.10). One quarter ahead, the last value is very hard to beat.

**Preferred forecast: about 0.57 %**, the inverse-MSE combination, with an 80 % range of about 0.50–0.66 % (the ARIMA and random-walk intervals). Why a combination, after what we saw in 4.3?
1. Section 4 showed that the ranking of single models is unstable.
2. The weakness of combinations is a common bias over long horizons. At $h = 1$ every model starts from the same last observation (0.58 %), so that bias has no time to build up.
3. The combination stays close to the random walk while using the information in the covariates.

## 6. Explainability
### 6.1 Dynamic multipliers of the econometric models
Response of $y$ to a +1 pp shock on unemployment: one-off shock (dynamic multipliers) and permanent shift (cumulated multipliers). For the regression with ARMA errors the answer is simply $\beta$ at every horizon.

In [ ]:
def dynamic_multipliers(params, p, var, q, horizon=12):
    phi = [params.get(f"y.L{i}", 0) for i in range(1, p + 1)]
    beta = [params.get(f"{var}.L{l}", 0) for l in range(q + 1)]
    m = np.zeros(horizon + 1)
    for t in range(horizon + 1):
        m[t] = (beta[t] if t <= q else 0) + sum(phi[i] * m[t - 1 - i] for i in range(p) if t - 1 - i >= 0)
    return pd.DataFrame({"one-off shock": m, "permanent shift": m.cumsum()})


fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for var, q in ARDL_Q.items():
    dm = dynamic_multipliers(ardl_res.params, ARDL_P, var, q)
    axes[0].plot(dm["one-off shock"], "o-", ms=3, label=f"ARDL: {var}")
    axes[1].plot(dm["permanent shift"], "o-", ms=3, label=f"ARDL: {var}")
for c in ARIMAX_COLS:
    axes[1].axhline(arimax_res.params[c], color="grey", ls=":", label=f"reg. with ARMA errors: β_{c}")
axes[0].set_title("One-off +1 pp shock")
axes[1].set_title("Permanent +1 pp shift")
for ax in axes:
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("quarters")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### 6.2 SHAP values of the LightGBM model (Darts `ShapExplainer`)

In [ ]:
from darts.explainability import ShapExplainer

lgb_full = lightgbm(LGB_BEST.L, LGB_BEST["cov lags"], LGB_BEST.min_leaf, LGB_BEST.trees)(SEED)
lgb_full.fit(to_ts(insample["y"]), future_covariates=to_ts(insample[X]))
explainer = ShapExplainer(lgb_full, background_series=to_ts(insample["y"]), background_future_covariates=to_ts(insample[X]))
shap_values = explainer.explain().get_explanation(horizon=1).to_dataframe()
print("Mean |SHAP| (percentage points of charge-off rate):")
print(shap_values.abs().mean().sort_values(ascending=False).round(4).to_string())
_ = explainer.summary_plot(horizons=[1])  # returns the raw SHAP arrays, not needed here

### 6.3 Ablation: what happens to the backtest MASE when a covariate is removed?

In [ ]:
ablation_specs = {
    "ARIMAX (all x)": lambda cols: arima_model(ARIMA_ORDER, cols or None),
    "Darts-LR": lambda cols: darts_model(linreg(LR_BEST.L, LR_BEST["cov lags"] if cols else None), cols),
    "Darts-LightGBM": lambda cols: darts_model(lightgbm(LGB_BEST.L, LGB_BEST["cov lags"] if cols else None, LGB_BEST.min_leaf, LGB_BEST.trees), cols),
}
if TFM_OK:
    ablation_specs["TimesFM-XReg"] = lambda cols: timesfm_xreg(cols, TFM_BEST["ridge"]) if cols else timesfm_zero_shot

ablation = {}
for name, make in ablation_specs.items():
    row = {"all four": scores(backtest(make(X)))["MASE"]}
    for drop in X:
        row[f"without {drop}"] = scores(backtest(make([c for c in X if c != drop])))["MASE"]
    row["no covariate"] = scores(backtest(make([])))["MASE"]
    ablation[name] = row
ablation = pd.DataFrame(ablation).T
ablation.round(3)

### 6.4 Sensitivity of the 2024–2025 forecasts to the WEO scenario
We shift the whole WEO path of one covariate and look at the average change of the 8 forecasts.

In [ ]:
shocks = {"u +1 pp": ("u", 1.0), "g -1 pp": ("g", -1.0), "pi +1 pp": ("pi", 1.0), "ca -1 pp": ("ca", -1.0)}
sensitivity = {}
for m, f in MODELS.items():
    if m in ("Naive (RW)", f"ARIMA{ARIMA_ORDER}", "TimesFM zero-shot"):
        continue
    sensitivity[m] = {}
    for label, (col, shift) in shocks.items():
        shocked = scen23.copy()
        shocked[col] += shift
        sensitivity[m][label] = (f(insample, shocked, H)["mean"] - fc_ante[m]["mean"]).mean()
pd.DataFrame(sensitivity).T.round(4)

**Multipliers (econometric models).**
* In the ARDL, a one-off +1 pp shock on unemployment raises the rate by 1.75 bp on impact. The effect then turns negative and oscillates along the credit cycle.
* A permanent +1 pp shift converges to −2.9 bp, which is not significant.
* In the regression with ARMA errors, the effect is +1.3 bp at every horizon, by construction.

**SHAP (LightGBM).** The last value of $y$ dominates (mean |SHAP| 0.068 pp). It is followed by growth at $t$ (0.017), unemployment at $t-1$ (0.015) and the 4th lag of $y$. Inflation and the current account have the smallest contributions (< 0.005).

**Ablation.**
* *ARIMAX, starting from all four covariates:* removing $\pi$ or $ca$ *improves* the backtest, and the model with no covariate at all (2.16) is as good as the best subset.
* *Darts-LR:* removing $g$ makes it much worse (3.38 → 5.04), while removing $ca$ makes it better.
* *LightGBM:* it barely moves (2.25–2.54), because trees simply ignore weak features. Without any covariate it gets a bit worse (2.54), so the covariates help a little.

**Scenario sensitivity.**
* Lower growth (−1 pp) raises the forecasts of the Darts models by about 4 bp, which is the expected sign.
* Higher unemployment (+1 pp) *lowers* the forecasts of the ARDL, Darts-LR and LightGBM (−1 to −4 bp). Only the ARIMAX has the intuitive positive sign.

**Do ML and econometrics agree? Yes.** All methods agree that $g$ and $u$ are the only useful covariates and that $\pi$ and $ca$ are noise. This matches their weak significance in the ARIMAX and their absence from the BIC ARDL.

**Is it economically meaningful?** Only half.
* The growth effect makes sense: recessions bring defaults.
* The negative long-run effect of unemployment does not. It is a Covid artefact: in 2020–21 unemployment was very high while charge-offs collapsed thanks to PPP loans and fiscal transfers. Every model estimated on 2012–2023 has learned this one episode.

This is the deeper reason why the covariates did not help out of sample. A credit-cycle variable (lending standards from the Fed's SLOOS, or the corporate bond spread) would probably be more informative than the macro aggregates we were given.

## 7. AI usage statement
**Tools.** We used an AI assistant (Claude, Anthropic) as a pair-programmer: to draft code, to look up library interfaces, to debug error messages and to reread our explanations. Every modelling choice, every number quoted in the text and every interpretation was checked by us against the outputs of this notebook and the official documentation of the installed versions (printed in section 0).

**Where the AI was wrong, and how we caught it.**
1. **ARDL lag orders.** The first version read the lags selected by `ardl_select_order` by zipping a list of orders with the covariate names. But `dl_lags` is a *dictionary*, and covariates dropped by BIC are absent from it, so the lags were assigned to the wrong variables. We noticed because the printed orders did not match the coefficients in the summary. We now read `sel.model.dl_lags` directly.
2. **Bounds test.** The assistant used `bounds.stat` and `bounds.p_values`, which raised an `AttributeError`. The statsmodels source shows that the fields are `statistic` and `pvalue`.
3. **TimesFM API.** It first proposed the TimesFM 1.x interface (`TimesFm(hparams=..., checkpoint=...)`), which does not exist in 2.5. Version 2.5 uses `TimesFM_2p5_200M_torch.from_pretrained(...)` followed by `compile(ForecastConfig(...))`. We also found in the source that `forecast_with_covariates` fails unless `return_backcast=True`, and that the first column of the quantile output is the mean, not the 10 % quantile.
4. **TimesFM backcast (the most dangerous one, because it gave no error).** Once `return_backcast=True` is set, `forecast()` returns the in-context backcast *followed by* the forecast. The code first proposed for us took the first $h$ values, so the "forecasts" were in fact fitted values of the past. We caught it by running the pipeline with randomly initialised weights before downloading the checkpoint: an 8-quarter forecast came back with 40 values. The forecast is the *last* $h$ values. We also found that the `"timesfm + xreg"` mode crashes when the history is shorter than one patch (32 quarters), which is why we only use `"xreg + timesfm"`.
5. **Small-sample traps.**
   * The default lag search of the Zivot–Andrews test is infeasible with 48 points (we set `max_lags=4`).
   * A Ljung–Box test at 4 lags on an ARMA(3,1) has zero degrees of freedom and returns `NaN` (we test at 6, 8 and 12 lags).
   * Darts rejects hyperparameters stored as `numpy.int64` (we cast them to `int`).

**Lesson.** The assistant was fast and useful, but it was confidently wrong whenever an interface had changed between versions. It never flagged an econometric issue on its own, such as the degenerate case of the bounds test. Running the code, printing intermediate objects and reading the source were our main safeguards.